In [1]:
import pandas as pd

crimes = pd.read_parquet("data/crimes_clean.parquet")
lookup = pd.read_parquet("data/community_lookup.parquet")

# attach area name + socioeconomic context to every crime
df = crimes.merge(lookup, on="community_area", how="left")

print(f"{len(df):,} rows after join")
# check the join worked — should be very few/zero unmatched
print("Rows with no area name:", df["area_name"].isna().sum())
df[["community_area", "area_name", "crime_category", "per_capita_income"]].head()

1,476,040 rows after join
Rows with no area name: 0


,community_area,area_name,crime_category,per_capita_income
0,66,Chicago Lawn,Violent,13231
1,66,Chicago Lawn,Violent,13231
2,67,West Englewood,Property,11317
3,25,Austin,Other,15957
4,58,Brighton Park,Violent,13089


In [2]:
import os
os.makedirs("data/tableau", exist_ok=True)

In [3]:
area_table = (
    df.groupby(["community_area", "area_name", "crime_category"])
      .agg(
          crimes=("id", "size"),
          arrests=("arrest", "sum"),
      )
      .reset_index()
)

# attach the socioeconomic context (one row per area, so take first)
context = df.groupby("community_area")[
    ["pct_below_poverty", "per_capita_income", "hardship_index"]
].first().reset_index()

area_table = area_table.merge(context, on="community_area", how="left")
area_table["arrest_rate"] = (area_table["arrests"] / area_table["crimes"]).round(3)

area_table.to_csv("data/tableau/crimes_by_area.csv", index=False)
print(f"crimes_by_area.csv — {len(area_table)} rows")
area_table.head()

crimes_by_area.csv — 308 rows


,community_area,area_name,crime_category,crimes,arrests,pct_below_poverty,per_capita_income,hardship_index,arrest_rate
0,1,Rogers Park,Other,3170,225,23.6,23939,39.0,0.071
1,1,Rogers Park,Property,12482,1217,23.6,23939,39.0,0.098
2,1,Rogers Park,Quality-of-Life,836,667,23.6,23939,39.0,0.798
3,1,Rogers Park,Violent,7822,1295,23.6,23939,39.0,0.166
4,2,West Ridge,Other,3689,165,17.2,23040,46.0,0.045


In [4]:
# order days properly so Tableau doesn't sort them alphabetically
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
             "Friday", "Saturday", "Sunday"]

time_table = (
    df.groupby(["dayofweek", "hour"])
      .agg(crimes=("id", "size"))
      .reset_index()
)
time_table["dayofweek"] = pd.Categorical(time_table["dayofweek"],
                                         categories=day_order, ordered=True)
time_table = time_table.sort_values(["dayofweek", "hour"])

time_table.to_csv("data/tableau/crimes_by_time.csv", index=False)
print(f"crimes_by_time.csv — {len(time_table)} rows")  # ~168 (7 days x 24 hrs)
time_table.head()

crimes_by_time.csv — 168 rows


,dayofweek,hour,crimes
24,Monday,0,14469
25,Monday,1,6536
26,Monday,2,5615
27,Monday,3,4644
28,Monday,4,3819


In [5]:
trend_table = (
    df.groupby(["year", "month", "crime_category"])
      .agg(crimes=("id", "size"))
      .reset_index()
)
# a real date for clean time-axis plotting in Tableau
trend_table["year_month"] = pd.to_datetime(
    trend_table["year"].astype(str) + "-" +
    trend_table["month"].astype(str) + "-01"
)

trend_table.to_csv("data/tableau/crimes_trend.csv", index=False)
print(f"crimes_trend.csv — {len(trend_table)} rows")
trend_table.head()

crimes_trend.csv — 300 rows


,year,month,crime_category,crimes,year_month
0,2019,1,Other,2939,2019-01-01
1,2019,1,Property,8383,2019-01-01
2,2019,1,Quality-of-Life,2072,2019-01-01
3,2019,1,Violent,6156,2019-01-01
4,2019,2,Other,2729,2019-02-01


In [6]:
# 75k random points is plenty for a convincing density map, keeps the file small
sample = df.sample(n=75_000, random_state=42)[
    ["latitude", "longitude", "crime_category", "year", "arrest"]
]
sample.to_csv("data/tableau/crime_points_sample.csv", index=False)

size_kb = os.path.getsize("data/tableau/crime_points_sample.csv") / 1e3
print(f"crime_points_sample.csv — {len(sample):,} rows ({size_kb:.0f} KB)")
print("\nAll Tableau files:", os.listdir("data/tableau"))

crime_points_sample.csv — 75,000 rows (3480 KB)

All Tableau files: ['crimes_by_area.csv', 'crimes_by_time.csv', 'crimes_trend.csv', 'crime_points_sample.csv']
